# Celebrity Detection + Dynamic Tags / Concepts Workflow

This notebook provides a **reusable, end-to-end workflow** for joining Coactive's celebrity face detection data with Dynamic Tag (DT) or Concept scores -- **entirely via the Coactive API**.

## Prerequisites
1. **Coactive API key** from your Coactive account
2. **Celebrities enrolled** via the enrollment script (see README) and backfill complete (~1-2 hours)
3. **Dynamic Tags and/or Concepts** created and published on your dataset in the Coactive UI

## What This Notebook Does
1. Authenticates with the Coactive API
2. Fetches celebrity face detections via the Celebrity Detection API
3. Fetches Concept scores and/or DT scores via the Query Engine SQL API
4. Joins faces + scores on `image_id`
5. Produces analytics: per-celebrity stats, cross-tab breakdowns, top keyframes
6. Generates Coactive UI SQL queries for visual verification

## SQL Tables & Schemas
| Table | Schema | Score Column | Notes |
|-------|--------|-------------|-------|
| `coactive_table_adv` | **Pivoted** -- one column per concept | `<concept>_prob` (e.g., `baton_prob`) | 0-1 probability |
| `dt_[group_name]_visual` | **Normalized** -- one row per (image, tag) | `tag_name` + `score` | 0-1 score, table named after your DT group |

You can use **Concepts only**, **DTs only**, or **both** -- configure in Step 0.

---
## Step 0: Configuration

Fill in your dataset, celebrity, and DT/Concept details below.

In [ ]:
# ============================================================
# CONFIGURATION - Update these values for your use case
# ============================================================

# --- Required ---
COACTIVE_API_KEY = ""  # Your Coactive API key (or set env var COACTIVE_API_KEY)
DATASET_ID = ""        # Your Coactive dataset ID (UUID)
BASE_URL = "https://app.coactive.ai"  # Coactive API base URL

# --- Celebrity Person IDs (from enrollment) ---
# Format: {"Display Name": "person_id_uuid"}
# You get person_ids after running the enrollment script.
# To list enrolled persons: python celebrity_enrollment.py --list
CELEBRITIES = {
    # "Celebrity Name": "person-id-uuid-here",
    # Example:
    # "Stephen Curry": "fa5a425f-f873-4a10-b9ba-66dcc1168357",
    # "LeBron James": "7c09dc53-d4cb-4b6e-96e4-f8a05ecbceee",
}

# --- Concepts (if using Concepts via Query Engine SQL) ---
# Each trained concept gets its own probability column in coactive_table_adv.
# Column names follow the pattern: <concept_name>_prob (e.g., "baton_prob").
# To discover your column names, run: SELECT * FROM coactive_table_adv LIMIT 1
#
# Format: {"Display Name": "sql_column_name_in_coactive_table_adv"}
CONCEPTS = {
    # "Baton": "baton_prob",
    # "Logo": "logo_prob",
}

# --- Dynamic Tags via SQL (RECOMMENDED - primary approach) ---
# Each DT group gets a SQL table named dt_[group_name]_visual (spaces become underscores).
# Schema: coactive_image_id | tag_name | score (normalized 0-1)
# To discover your table name, run: SHOW TABLES in the Coactive SQL editor.
#
# Format: {"Display Name": "sql_table_name"}
DT_SQL_TABLES = {
    # "Sports Classification": "dt_sports_classification_visual",
    # "Broadcast Context": "dt_broadcast_context_visual",
}

# --- Dynamic Tags via API (FALLBACK - use only if SQL tables unavailable) ---
# Get group/version IDs from the Coactive UI or API
# Only needed if DT_SQL_TABLES is empty and you want to use DT scores via the scoring API
DT_GROUPS = [
    # {
    #     "name": "Your Tag Group Name",
    #     "group_id": "uuid-here",
    #     "group_version_id": "uuid-here",
    # },
]

# --- Thresholds ---
FACE_CONFIDENCE_THRESHOLD = 0.7   # Min face detection confidence
CONCEPT_SCORE_THRESHOLD = 0.5     # Min concept probability (0-1)
DT_SCORE_THRESHOLD = 0.5          # Min DT tag score (0-1, from SQL or scoring-preview)
DT_COSINE_THRESHOLD = 0.19        # Min raw cosine similarity (only used with asset-check API fallback)

# ============================================================
import os
if not COACTIVE_API_KEY:
    COACTIVE_API_KEY = os.environ.get("COACTIVE_API_KEY", "")

assert COACTIVE_API_KEY, "Set COACTIVE_API_KEY above or via environment variable"
assert DATASET_ID, "Set DATASET_ID above"
assert CELEBRITIES, "Add at least one celebrity to CELEBRITIES dict"
assert CONCEPTS or DT_SQL_TABLES or DT_GROUPS, "Configure at least one of CONCEPTS, DT_SQL_TABLES, or DT_GROUPS"

print("Configuration loaded.")
print(f"  Dataset: {DATASET_ID}")
print(f"  Celebrities: {list(CELEBRITIES.keys())}")
print(f"  Concepts: {list(CONCEPTS.keys()) if CONCEPTS else 'None'}")
print(f"  DT SQL tables: {list(DT_SQL_TABLES.keys()) if DT_SQL_TABLES else 'None'}")
print(f"  DT API groups: {[g['name'] for g in DT_GROUPS] if DT_GROUPS else 'None'}")
if DT_SQL_TABLES and DT_GROUPS:
    print("  Note: DT_SQL_TABLES takes priority. DT_GROUPS (API) will be skipped.")

---
## Step 1: Authenticate with Coactive API

In [ ]:
import requests
import json
import time
import csv
import io
import urllib3
from collections import defaultdict, Counter
from concurrent.futures import ThreadPoolExecutor, as_completed
from itertools import product as iterproduct

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

def get_jwt(api_key):
    """Exchange API key for JWT token."""
    r = requests.post(
        f"{BASE_URL}/api/v0/login",
        headers={"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"},
        json={"grant_type": "refresh_token"},
        verify=False, timeout=60,
    )
    r.raise_for_status()
    return r.json()["access_token"]

jwt_token = get_jwt(COACTIVE_API_KEY)
HEADERS = {"Authorization": f"Bearer {jwt_token}", "Content-Type": "application/json"}
print(f"Authenticated successfully (JWT: {jwt_token[:30]}...)")

---
## Step 2: Fetch Celebrity Face Detections

Pulls all detected faces for each enrolled celebrity from the Celebrity Detection API, filtered to your dataset.

In [ ]:
CELEB_BASE = f"{BASE_URL}/api/v0/celebrity-detection"

all_faces = []
for celeb_name, person_id in CELEBRITIES.items():
    r = requests.get(
        f"{CELEB_BASE}/faces-with-person",
        headers=HEADERS,
        params={"person_name": celeb_name},
        verify=False, timeout=120,
    )
    r.raise_for_status()
    faces = r.json().get("faces", [])
    dataset_faces = [f for f in faces if f.get("dataset_id") == DATASET_ID]

    for face in dataset_faces:
        all_faces.append({
            "image_id": face["image_id"],
            "celebrity_name": celeb_name,
            "person_id": person_id,
            "confidence": face.get("confidence", 0),
            "face_id": face.get("id"),
            "bbox": face.get("bbox", {}),
        })
    print(f"  {celeb_name}: {len(dataset_faces)} faces")

print(f"\nTotal faces loaded: {len(all_faces)}")

# Deduplicate: keep highest confidence per (image_id, celebrity_name)
best_faces = {}
for face in all_faces:
    key = (face["image_id"], face["celebrity_name"])
    if key not in best_faces or face["confidence"] > best_faces[key]["confidence"]:
        best_faces[key] = face

faces_deduped = list(best_faces.values())
face_image_ids = list(set(f["image_id"] for f in faces_deduped))
face_image_set = set(face_image_ids)

print(f"Unique (image, celebrity) pairs: {len(faces_deduped)}")
print(f"Unique keyframes with celebrity faces: {len(face_image_ids)}")

---
## Step 3A: Fetch Concept Scores via Query Engine (SQL API)

Uses the Coactive Query Engine to run SQL against `coactive_table_adv`, which contains per-image concept probability columns.

**Schema:** Each trained concept gets its own column (e.g., `baton_prob`). So the query is:
```sql
SELECT coactive_image_id, baton_prob, logo_prob
FROM coactive_table_adv
WHERE coactive_image_id IN ('id1', 'id2', ...)
  AND (baton_prob > 0.5 OR logo_prob > 0.5)
```

**How the Query Engine works:**
1. Submit SQL via `POST /api/v1/queries` (async)
2. Poll status via `GET /api/v1/queries/{query_id}`
3. Download results: `GET /api/v1/queries/{query_id}/results` → returns presigned `download_url` for CSV

Skip this step if you're only using Dynamic Tags.

In [ ]:
# --- Query Engine helper functions ---

def submit_query(sql, dataset_id):
    """Submit a SQL query to the Coactive Query Engine. Returns query_id."""
    r = requests.post(
        f"{BASE_URL}/api/v1/queries",
        headers=HEADERS,
        json={"query": sql, "dataset_id": dataset_id},
        verify=False, timeout=60,
    )
    r.raise_for_status()
    data = r.json()
    query_id = data.get("queryId") or data.get("query_id") or data.get("id")
    print(f"  Query submitted: {query_id}")
    return query_id


def poll_query(query_id, max_wait=300, interval=5):
    """Poll query status until complete. Returns result metadata."""
    elapsed = 0
    while elapsed < max_wait:
        r = requests.get(
            f"{BASE_URL}/api/v1/queries/{query_id}",
            headers=HEADERS,
            verify=False, timeout=30,
        )
        r.raise_for_status()
        data = r.json()
        status = data.get("status", "").lower()
        if status in ("completed", "complete", "succeeded", "success"):
            print(f"  Query complete ({elapsed}s)")
            return data
        elif status in ("failed", "error"):
            raise RuntimeError(f"Query failed: {data}")
        time.sleep(interval)
        elapsed += interval
    raise TimeoutError(f"Query {query_id} did not complete within {max_wait}s")


def download_query_results(query_id):
    """Download query results as list of dicts.

    Tries multiple download strategies:
    1. GET /results/csv -- direct CSV or JSON with downloadUrl
    2. GET /csv/url -- JSON with downloadUrl
    3. GET /results -- JSON with download_url (presigned S3 URL)
    """
    # Strategy 1: /results/csv endpoint
    try:
        r = requests.get(
            f"{BASE_URL}/api/v1/queries/{query_id}/results/csv",
            headers=HEADERS,
            verify=False, timeout=120,
        )
        if r.status_code == 200:
            content_type = r.headers.get("content-type", "")
            if "csv" in content_type or "text" in content_type:
                reader = csv.DictReader(io.StringIO(r.text))
                return list(reader)
            else:
                data = r.json()
                download_url = data.get("downloadUrl") or data.get("download_url") or data.get("url")
                if download_url:
                    r2 = requests.get(download_url, timeout=120)
                    r2.raise_for_status()
                    reader = csv.DictReader(io.StringIO(r2.text))
                    return list(reader)
    except Exception:
        pass

    # Strategy 2: /csv/url endpoint
    try:
        r = requests.get(
            f"{BASE_URL}/api/v1/queries/{query_id}/csv/url",
            headers=HEADERS,
            verify=False, timeout=30,
        )
        if r.status_code == 200:
            data = r.json()
            download_url = data.get("downloadUrl") or data.get("download_url") or data.get("url")
            if download_url:
                r2 = requests.get(download_url, timeout=120)
                r2.raise_for_status()
                reader = csv.DictReader(io.StringIO(r2.text))
                return list(reader)
    except Exception:
        pass

    # Strategy 3: /results endpoint (returns download_url for presigned S3 CSV)
    r = requests.get(
        f"{BASE_URL}/api/v1/queries/{query_id}/results",
        headers=HEADERS,
        verify=False, timeout=120,
    )
    r.raise_for_status()
    data = r.json()

    # Check for download URL (presigned S3 URL)
    download_url = data.get("download_url") or data.get("downloadUrl") or data.get("url")
    if download_url:
        r2 = requests.get(download_url, timeout=120)
        r2.raise_for_status()
        reader = csv.DictReader(io.StringIO(r2.text))
        return list(reader)

    # Fallback: paginated JSON results
    rows = data.get("results", data.get("data", data.get("rows", [])))
    columns = data.get("columns", [])
    if columns and rows and isinstance(rows[0], list):
        return [dict(zip(columns, row)) for row in rows]
    return rows


def run_query(sql, dataset_id):
    """Submit, poll, and download query results. Returns list of dicts."""
    query_id = submit_query(sql, dataset_id)
    poll_query(query_id)
    results = download_query_results(query_id)
    print(f"  Results: {len(results)} rows")
    return results

print("Query Engine helpers loaded.")

In [ ]:
# --- Fetch concept scores for all celebrity keyframes ---
# Uses coactive_table_adv where each concept is a column (e.g., baton_prob)

concept_scores = {}  # {image_id: {concept_display_name: probability}}

if CONCEPTS:
    print(f"Fetching concept scores for {len(face_image_ids)} keyframes...")
    print(f"Concepts: {CONCEPTS}\n")

    # Build column list from configured concepts
    concept_columns = list(CONCEPTS.values())  # e.g., ["baton_prob", "logo_prob"]
    concept_col_to_name = {v: k for k, v in CONCEPTS.items()}  # reverse map

    # Build the WHERE clause: at least one concept above threshold
    or_clauses = " OR ".join([f"{col} > {CONCEPT_SCORE_THRESHOLD}" for col in concept_columns])
    select_cols = ", ".join(["coactive_image_id"] + concept_columns)

    # Process in batches to avoid SQL length limits
    BATCH_SIZE = 200
    all_concept_rows = []

    for batch_start in range(0, len(face_image_ids), BATCH_SIZE):
        batch_ids = face_image_ids[batch_start:batch_start + BATCH_SIZE]
        ids_sql = ", ".join([f"'{iid}'" for iid in batch_ids])

        sql = f"""
        SELECT {select_cols}
        FROM coactive_table_adv
        WHERE coactive_image_id IN ({ids_sql})
          AND ({or_clauses})
        """

        batch_num = batch_start // BATCH_SIZE + 1
        total_batches = (len(face_image_ids) + BATCH_SIZE - 1) // BATCH_SIZE
        print(f"Batch {batch_num}/{total_batches} ({len(batch_ids)} keyframes):")

        try:
            rows = run_query(sql, DATASET_ID)
            all_concept_rows.extend(rows)
        except Exception as e:
            print(f"  Warning: batch {batch_num} failed: {e}")

    # Build concept_scores dict by unpivoting the columnar results
    for row in all_concept_rows:
        image_id = row.get("coactive_image_id", "")
        if not image_id:
            continue
        for col_name, display_name in concept_col_to_name.items():
            prob = row.get(col_name)
            if prob is not None:
                prob = float(prob)
                if prob > CONCEPT_SCORE_THRESHOLD:
                    concept_scores.setdefault(image_id, {})[display_name] = prob

    print(f"\nConcept scoring complete:")
    print(f"  Query result rows: {len(all_concept_rows)}")
    print(f"  Keyframes with concept scores: {len(concept_scores)}/{len(face_image_set)}")
    for display_name, col_name in CONCEPTS.items():
        count = sum(1 for scores in concept_scores.values() if display_name in scores)
        print(f"    {display_name} ({col_name}): {count} keyframes above threshold")
else:
    print("No CONCEPTS configured -- skipping. Using DT scores only.")

---
## Step 3B: Fetch Dynamic Tag Scores via SQL (Primary Approach)

Uses the Coactive Query Engine to run SQL against `dt_[group_name]_visual` tables, which have a normalized schema:

```sql
SELECT coactive_image_id, tag_name, score
FROM dt_sports_classification_visual
WHERE coactive_image_id IN ('id1', 'id2', ...)
  AND score > 0.01
```

Each DT group gets its own table (e.g., `dt_sports_classification_visual`). The table name is the group name lowercased with spaces replaced by underscores, prefixed with `dt_` and suffixed with `_visual`.

Skip this step if you're only using Concepts.

In [ ]:
# --- Fetch DT scores via SQL (primary) or API (fallback) ---

dt_scores = {}       # {image_id: {tag_name: score}}
score_source = {}    # {(image_id, tag_name): "dt-sql" | "preview" | "asset-check"}
all_tags = []        # tag metadata (populated by API path only)
dt_tag_names = []    # all discovered DT tag names (from SQL or API)

USE_DT_SQL = bool(DT_SQL_TABLES)
USE_DT_API = bool(DT_GROUPS) and not USE_DT_SQL  # API is fallback only

# ============================================================
# PATH A: SQL-based DT scoring (RECOMMENDED)
# ============================================================
if USE_DT_SQL:
    print(f"Fetching DT scores via SQL for {len(face_image_ids)} keyframes...")
    print(f"DT tables: {DT_SQL_TABLES}\n")

    BATCH_SIZE = 200

    for group_display_name, table_name in DT_SQL_TABLES.items():
        print(f"\n--- {group_display_name} (table: {table_name}) ---")
        all_dt_rows = []

        for batch_start in range(0, len(face_image_ids), BATCH_SIZE):
            batch_ids = face_image_ids[batch_start:batch_start + BATCH_SIZE]
            ids_sql = ", ".join([f"'{iid}'" for iid in batch_ids])

            sql = f"""
            SELECT coactive_image_id, tag_name, score
            FROM {table_name}
            WHERE coactive_image_id IN ({ids_sql})
              AND score > 0.01
            """

            batch_num = batch_start // BATCH_SIZE + 1
            total_batches = (len(face_image_ids) + BATCH_SIZE - 1) // BATCH_SIZE
            print(f"  Batch {batch_num}/{total_batches} ({len(batch_ids)} keyframes):")

            try:
                rows = run_query(sql, DATASET_ID)
                all_dt_rows.extend(rows)
            except Exception as e:
                print(f"  Warning: batch {batch_num} failed: {e}")

        # Build dt_scores dict from normalized rows
        for row in all_dt_rows:
            image_id = row.get("coactive_image_id", "")
            tag_name = row.get("tag_name", "")
            score_val = row.get("score")
            if not image_id or not tag_name or score_val is None:
                continue
            score_val = float(score_val)
            dt_scores.setdefault(image_id, {})[tag_name] = score_val
            score_source[(image_id, tag_name)] = "dt-sql"

        # Track discovered tag names
        group_tags = sorted(set(row.get("tag_name", "") for row in all_dt_rows if row.get("tag_name")))
        dt_tag_names.extend(group_tags)
        print(f"  Result rows: {len(all_dt_rows)}")
        print(f"  Tags found: {group_tags}")

    print(f"\nDT SQL scoring complete:")
    print(f"  Keyframes with DT scores: {len(dt_scores)}/{len(face_image_set)}")
    for tag in sorted(set(dt_tag_names)):
        count = sum(1 for img_scores in dt_scores.values() if tag in img_scores)
        hq_count = sum(1 for img_scores in dt_scores.values() if img_scores.get(tag, 0) > DT_SCORE_THRESHOLD)
        print(f"    {tag}: {count} total, {hq_count} above threshold ({DT_SCORE_THRESHOLD})")

# ============================================================
# PATH B: API-based DT scoring (FALLBACK)
# ============================================================
elif USE_DT_API:
    V3_DT_BASE = f"{BASE_URL}/api/v3/dynamic-tags"

    # --- Fetch DT tag metadata ---
    for group in DT_GROUPS:
        gid = group["group_id"]
        gvid = group["group_version_id"]
        r = requests.get(
            f"{V3_DT_BASE}/groups/{gid}/versions/{gvid}/tags?per_page=50",
            headers=HEADERS, verify=False, timeout=30,
        )
        r.raise_for_status()
        for t in r.json().get("data", []):
            tag_info = t.get("tag", {})
            ver_info = t.get("version", {})
            all_tags.append({
                "tag_name": ver_info.get("name"),
                "tag_id": tag_info.get("id"),
                "tag_version_id": ver_info.get("id"),
                "group_id": gid,
                "group_version_id": gvid,
                "group_name": group["name"],
            })
            print(f"  [{group['name']}] {ver_info.get('name')}")
    print(f"\nTotal DT tags: {len(all_tags)}")
    dt_tag_names = [t["tag_name"] for t in all_tags]

    # --- Phase 1: scoring-preview (actual DT scores, ~100/tag) ---
    print("\nPhase 1: scoring-preview...")
    for tag in all_tags:
        url = (f"{V3_DT_BASE}/groups/{tag['group_id']}/tags/{tag['tag_id']}"
               f"/versions/{tag['tag_version_id']}/scoring-preview/image-and-keyframe")
        r = requests.get(url, headers=HEADERS, params={"prompt_types": ["text", "visual"]},
                         verify=False, timeout=60)
        if r.status_code == 200:
            tag_scores_list = r.json().get("scores", [])
            hits = 0
            for entry in tag_scores_list:
                asset_id = entry.get("asset_id")
                if asset_id in face_image_set:
                    dt_scores.setdefault(asset_id, {})[tag["tag_name"]] = entry["score"]
                    score_source[(asset_id, tag["tag_name"])] = "preview"
                    hits += 1
            print(f"  {tag['tag_name']:35s}: {len(tag_scores_list)} scored, {hits} face hits")
        else:
            print(f"  {tag['tag_name']:35s}: HTTP {r.status_code}")

    print(f"\nPhase 1: {len(dt_scores)}/{len(face_image_set)} keyframes with scores")

else:
    print("No DT_SQL_TABLES or DT_GROUPS configured -- skipping. Using Concept scores only.")

In [ ]:
# --- Phase 2: asset-check fallback (only for API path) ---
# This cell only runs when using DT_GROUPS (API fallback), NOT when using DT_SQL_TABLES.
# The SQL approach returns complete scores for all matching keyframes, so no fallback is needed.

if USE_DT_API and all_tags:
    V3_DT_BASE = f"{BASE_URL}/api/v3/dynamic-tags"

    def check_asset_score(tag, keyframe_id):
        """Get average cosine similarity for a keyframe against a tag's prompts."""
        url = (f"{V3_DT_BASE}/groups/{tag['group_id']}/tags/{tag['tag_id']}"
               f"/versions/{tag['tag_version_id']}/asset-check")
        try:
            r = requests.get(
                url, headers=HEADERS,
                params={"id": keyframe_id, "type": "keyframe"},
                verify=False, timeout=30,
            )
            if r.status_code == 200:
                prompts = (r.json()
                           .get("modality", {})
                           .get("visual", {})
                           .get("prompt_similarities", {})
                           .get("positive_prompts", []))
                sims = []
                for p in prompts:
                    for sim in p.get("similarity", []):
                        if sim.get("keyframe_id") == keyframe_id:
                            sims.append(sim.get("score", 0))
                return sum(sims) / len(sims) if sims else None
        except Exception:
            pass
        return None

    def score_one(tag, keyframe_id):
        return keyframe_id, tag["tag_name"], check_asset_score(tag, keyframe_id)

    # Find (keyframe, tag) pairs not yet scored
    missing_pairs = [
        (tag, kid) for kid in face_image_ids for tag in all_tags
        if (kid, tag["tag_name"]) not in score_source
    ]

    if missing_pairs:
        print(f"Phase 2: Scoring {len(missing_pairs)} remaining pairs via asset-check (10 workers)...")

        completed = 0
        with ThreadPoolExecutor(max_workers=10) as executor:
            futures = [executor.submit(score_one, tag, kid) for tag, kid in missing_pairs]
            for future in as_completed(futures):
                keyframe_id, tag_name, score = future.result()
                if score is not None:
                    dt_scores.setdefault(keyframe_id, {})[tag_name] = score
                    score_source[(keyframe_id, tag_name)] = "asset-check"
                completed += 1
                if completed % 500 == 0:
                    print(f"  Progress: {completed}/{len(missing_pairs)} ({100*completed/len(missing_pairs):.0f}%)")

        preview_count = sum(1 for v in score_source.values() if v == "preview")
        check_count = sum(1 for v in score_source.values() if v == "asset-check")
        print(f"\nDT API scoring complete!")
        print(f"  From scoring-preview: {preview_count}")
        print(f"  From asset-check: {check_count}")
        print(f"  Keyframes with DT scores: {len(dt_scores)}/{len(face_image_set)}")
    else:
        print("All (keyframe, tag) pairs already scored in Phase 1.")
elif USE_DT_SQL:
    print("Using SQL-based DT scoring -- asset-check fallback not needed.")
else:
    print("No DT scoring configured -- skipping.")

---
## Step 4: Join Celebrities + Scores

Combines face detections with concept and/or DT scores into a single joined dataset.

In [ ]:
joined_rows = []

for face in faces_deduped:
    image_id = face["image_id"]

    # Add concept score rows
    if image_id in concept_scores:
        for concept_name, probability in concept_scores[image_id].items():
            joined_rows.append({
                "image_id": image_id,
                "celebrity_name": face["celebrity_name"],
                "person_id": face["person_id"],
                "face_confidence": face["confidence"],
                "tag_name": concept_name,
                "tag_score": probability,
                "score_source": "concept",
            })

    # Add DT score rows
    if image_id in dt_scores:
        for tag_name, tag_score in dt_scores[image_id].items():
            joined_rows.append({
                "image_id": image_id,
                "celebrity_name": face["celebrity_name"],
                "person_id": face["person_id"],
                "face_confidence": face["confidence"],
                "tag_name": tag_name,
                "tag_score": tag_score,
                "score_source": score_source.get((image_id, tag_name), "dt"),
            })

print(f"Joined rows: {len(joined_rows)}")
source_counts = Counter(r["score_source"] for r in joined_rows)
print(f"Score sources: {dict(source_counts)}")

# Face counts per celebrity
face_counts = Counter(f["celebrity_name"] for f in faces_deduped)
print(f"\nFace counts:")
for celeb, cnt in face_counts.most_common():
    print(f"  {celeb}: {cnt}")

# Overlap stats
faces_with_concepts = len(set(f["image_id"] for f in faces_deduped if f["image_id"] in concept_scores))
faces_with_dt = len(set(f["image_id"] for f in faces_deduped if f["image_id"] in dt_scores))
faces_with_any = len(set(r["image_id"] for r in joined_rows))
print(f"\nScore overlap:")
print(f"  Faces with concept scores: {faces_with_concepts}/{len(face_image_set)}")
print(f"  Faces with DT scores: {faces_with_dt}/{len(face_image_set)}")
print(f"  Faces with any score: {faces_with_any}/{len(face_image_set)}")

# Top rows
print("\nTop 15 by tag_score:")
for row in sorted(joined_rows, key=lambda x: -x["tag_score"])[:15]:
    print(f"  {row['celebrity_name']:20s} | {row['tag_name']:30s} | conf={row['face_confidence']:.3f} | score={row['tag_score']:.4f} | {row['score_source']}")

---
## Step 5: Analytics & Breakdowns

In [ ]:
if not joined_rows:
    print("No joined rows -- check that face detections and scores overlap.")
else:
    tag_names = sorted(set(r["tag_name"] for r in joined_rows))
    celeb_names = sorted(CELEBRITIES.keys())

    # Determine threshold per source
    def is_high_quality(row):
        if row["face_confidence"] <= FACE_CONFIDENCE_THRESHOLD:
            return False
        if row["score_source"] == "concept":
            return row["tag_score"] > CONCEPT_SCORE_THRESHOLD
        elif row["score_source"] == "asset-check":
            return row["tag_score"] > DT_COSINE_THRESHOLD
        else:
            # dt-sql and preview both use normalized 0-1 scores
            return row["tag_score"] > DT_SCORE_THRESHOLD

    hq = [r for r in joined_rows if is_high_quality(r)]

    celeb_tag_rows = defaultdict(list)
    for r in hq:
        celeb_tag_rows[(r["celebrity_name"], r["tag_name"])].append(r)

    # --- Cross-tab ---
    print("=" * 90)
    print("CLASSIFICATION BREAKDOWN (high-quality matches)")
    print("=" * 90)

    # Determine tag grouping
    tag_to_group = {}
    if all_tags:
        tag_to_group = {t["tag_name"]: t["group_name"] for t in all_tags}
    if DT_SQL_TABLES:
        for group_name, table_name in DT_SQL_TABLES.items():
            for tag in dt_tag_names:
                if tag not in tag_to_group:
                    tag_to_group[tag] = group_name
    for cname in CONCEPTS:
        tag_to_group[cname] = "Concepts"

    for group_name in sorted(set(tag_to_group.values())):
        group_tags = [t for t in tag_names if tag_to_group.get(t) == group_name]
        if not group_tags:
            continue
        print(f"\n--- {group_name} ---")
        # Header
        last_names = [c.split()[-1] for c in celeb_names]
        header = f"{'Tag':<30s}" + "".join(f" | {n:>10s}" for n in last_names) + f" | {'TOTAL':>8s}"
        print(header)
        print("-" * len(header))
        for tag in group_tags:
            counts = [len(celeb_tag_rows.get((c, tag), [])) for c in celeb_names]
            total = sum(counts)
            row_str = f"{tag:<30s}" + "".join(f" | {c:>10d}" for c in counts) + f" | {total:>8d}"
            print(row_str)

    # --- Per-celebrity content mix ---
    print(f"\n\n{'=' * 90}")
    print("PER-CELEBRITY CONTENT MIX (% of high-quality matches)")
    print("=" * 90)
    for celeb in celeb_names:
        celeb_total = sum(len(celeb_tag_rows.get((celeb, t), [])) for t in tag_names)
        if celeb_total == 0:
            print(f"\n  {celeb}: No high-quality matches")
            continue
        print(f"\n  {celeb} ({celeb_total} classified keyframes):")
        for tag in tag_names:
            n = len(celeb_tag_rows.get((celeb, tag), []))
            if n > 0:
                pct = 100 * n / celeb_total
                bar = "#" * int(pct / 2)
                print(f"    {tag:<30s}: {n:>4d} ({pct:>5.1f}%) {bar}")

---
## Step 6: Generate Coactive UI Queries

Paste these SQL queries into the Coactive platform UI to visually browse matched keyframes.

In [ ]:
def generate_query(image_ids, label=""):
    """Print a Coactive UI SQL query for a list of image IDs."""
    ids_str = ", ".join([f"'{iid}'" for iid in image_ids[:15]])
    if label:
        print(f"\n-- {label}:")
    print(f"SELECT * FROM coactive_table WHERE coactive_image_id IN ({ids_str})")

if joined_rows:
    tag_names = sorted(set(r["tag_name"] for r in joined_rows))

    # --- Top 10 per Celebrity (by face confidence) ---
    print("=" * 80)
    print("TOP KEYFRAMES PER CELEBRITY (by face confidence)")
    print("=" * 80)
    for celeb_name in CELEBRITIES.keys():
        celeb_faces = sorted(
            [f for f in faces_deduped if f["celebrity_name"] == celeb_name],
            key=lambda x: -x["confidence"]
        )[:10]
        if celeb_faces:
            generate_query([f["image_id"] for f in celeb_faces],
                          f"{celeb_name} top 10 (avg_conf={sum(f['confidence'] for f in celeb_faces)/len(celeb_faces):.3f})")

    # --- Top 10 per Celebrity + Tag ---
    print(f"\n\n{'=' * 80}")
    print("TOP KEYFRAMES PER CELEBRITY + TAG (by tag_score)")
    print("=" * 80)
    for celeb_name, tag_name in iterproduct(CELEBRITIES.keys(), tag_names):
        matching = sorted(
            [r for r in joined_rows
             if r["celebrity_name"] == celeb_name
             and r["tag_name"] == tag_name
             and r["face_confidence"] > FACE_CONFIDENCE_THRESHOLD],
            key=lambda x: (-x["tag_score"], -x["face_confidence"])
        )[:10]
        if matching:
            seen = set()
            unique_ids = [r["image_id"] for r in matching if r["image_id"] not in seen and not seen.add(r["image_id"])]
            avg_score = sum(r["tag_score"] for r in matching) / len(matching)
            generate_query(unique_ids, f"{celeb_name} + {tag_name} (avg={avg_score:.3f}, n={len(unique_ids)})")

    # --- Per-athlete content profile (best keyframe per tag) ---
    print(f"\n\n{'=' * 80}")
    print("CONTENT PROFILE PER CELEBRITY (best keyframe per tag)")
    print("=" * 80)
    for celeb_name in CELEBRITIES.keys():
        best_per_tag = []
        for tag in tag_names:
            matches = sorted(
                [r for r in joined_rows
                 if r["celebrity_name"] == celeb_name and r["tag_name"] == tag
                 and r["face_confidence"] > FACE_CONFIDENCE_THRESHOLD],
                key=lambda x: (-x["tag_score"], -x["face_confidence"])
            )
            if matches:
                best_per_tag.append(matches[0])
        if best_per_tag:
            seen = set()
            unique_ids = [r["image_id"] for r in best_per_tag if r["image_id"] not in seen and not seen.add(r["image_id"])]
            generate_query(unique_ids, f"{celeb_name} content profile ({len(unique_ids)} tags)")

---
## Step 7: Export (Optional)

Export joined data as CSV for downstream use, or convert to a PySpark DataFrame if in Databricks.

In [ ]:
# --- CSV export ---
if joined_rows:
    output = io.StringIO()
    fieldnames = ["image_id", "celebrity_name", "person_id", "face_confidence", "tag_name", "tag_score", "score_source"]
    writer = csv.DictWriter(output, fieldnames=fieldnames)
    writer.writeheader()
    for row in sorted(joined_rows, key=lambda x: (x["celebrity_name"], x["tag_name"], -x["tag_score"])):
        writer.writerow(row)
    csv_data = output.getvalue()
    print(f"CSV export: {len(joined_rows)} rows, {len(csv_data):,} bytes")
    print("\nFirst 10 rows:")
    for line in csv_data.split('\n')[:11]:
        print(line)

# --- PySpark DataFrame (Databricks only) ---
if joined_rows:
    try:
        from pyspark.sql import SparkSession
        from pyspark.sql.functions import col, desc, count, avg, max as spark_max, row_number
        from pyspark.sql.window import Window

        spark = SparkSession.builder.getOrCreate()
        df_joined = spark.createDataFrame(joined_rows)
        print(f"\nPySpark DataFrame: {df_joined.count()} rows")
        df_joined.groupBy("celebrity_name", "score_source").agg(
            count("*").alias("count"),
            avg("face_confidence").alias("avg_conf"),
            avg("tag_score").alias("avg_score"),
        ).orderBy("celebrity_name").show()
    except ImportError:
        print("\nPySpark not available. Joined data is in the 'joined_rows' list.")